***
# **Parte C: Redes Neuronales Artificiales**

## Perceptrón

* El preceptrón es una estructura que trata de imitar el funcionamiento de una neurona.

* Siendo $\overline{x}=(x_{1}, x_{2},...,x_{n})$ nuestras entradas y $\overline{w}=(w_{1}, w_{2},...,w_{n})$ los pesos sinápticos, la salida del perceptron se calcula como:

$$perceptron(\overline{x})=f(\overline{x} \cdot \overline{w} + b)=f(\sum(x_{i} \cdot w_{i} )+ b)$$

* Como vimos en la teoría, el perceptron es similar a una regresión lineal ($f(z)=z$)  o a una regresión logística $f(z)=\frac{1}{1+e^{-z}}$, con la diferencia que reemplaza la función de activación **lineal** o **sigmoide**, respectivamente, por una de tipo **step**. Obviamente, existen otras funciones de activación que se irán discutiendo a lo largo del curso.


***
## Perceptrón multicapas

* Combinando varios preceptrones en forma paralela se forma una **capa** de una red neuronal. En este caso, la capa se conoce como *capa densa* ya que todas las salidas de la capa están conectadas con cada entrada. 

* En el caso de una capa, $W$ es una matriz de dimensiones $m$ x $n$, donde $m$ es la cantidad de features (características o entradas) y $n$ la cantidad de neuronas, $b$ es un vector de dimensionalidad igual a la cantidad de perceptrones, y la función de activación $f(z)$ se aplica a la salida de cada neurona. 

* Considerando el dataset MNIST, $W$ es una matriz de 784 X 10, y $b$ es un vector de 10 elementos.

* Cuando existe una o más **capas ocultas** de perceptrones entre la capa de entrada y la de salida, la red neuronal artificial (RNA) se conoce con el nombre de **Perceptron multicapa** o Multi-Layer Perceptron (MLP)

***

## Frameworks TensorFlow-Keras

* Los principales frameworks que se utilizan en la práctica para entrenar redes neuronales son [TensorFlow](https://www.tensorflow.org/), [PyTorch](https://pytorch.org/) y [MXNet Apache](https://mxnet.apache.org/versions/1.9.1/). 

* Ellos se encargan de calcular *automáticamente*
 las derivadas parciales de las funciones de pérdida para cada uno de los parámetros de nuestros modelos. 

* Empezaremos trabajando con [Keras](https://keras.io/) que es una librería de alto nivel para redes neuronales que abstrae las operaciones más comunes,  facilitando la escritura de un código más limpio. 

* Keras está construida sobre [TensorFlow](https://keras.io/backend/), por lo que no es necesario instalar ningún elemento extra. Todas las abstracciones de Keras se encuentran en el paquete tensorflow.keras.

A continuación, se muesta un ejemplo de utilización de Keras para el problema clasificación de MNIST.

In [ ]:
from tensorflow.keras.layers import Input, Dense 
from tensorflow.keras.models import Model 
from tensorflow.keras.optimizers import SGD 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.utils import to_categorical 
import numpy as np


A continuación se carga el dataset y se realiza un preprocesado a los datos (reshape, normalización, one-hot encoding, cambio de tipo)

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(f'x_train shape: {x_train.shape}')  #60000 x 28 x 28

size = x_train.shape[1]*x_train.shape[2] #tamaño de mi vector de entrada (hacemos un flattering a las imágenes) => 28x28 => 784
print(f'size: {size}') 
print(f'cantidad de muestras: {x_train.shape[0]}') #60000

#reshapeamos y normalizamos ([0 , 255] --> [0, 1])
x_train = x_train.reshape((x_train.shape[0], size)) / 255  
x_test = x_test.reshape((x_test.shape[0], size)) / 255
yc_train, yc_test = to_categorical(y_train), to_categorical(y_test)

#convierto de float64 a float32
x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)
yc_train = yc_train.astype(np.float32)
yc_test = yc_test.astype(np.float32)

In [ ]:
BS = 500 #batch-size

#usamos la API funcional de keras
i = Input((size,))
d = Dense(10, activation='softmax')(i)

model = Model(inputs=i, outputs=d)
model.compile(loss='categorical_crossentropy', optimizer='sgd')
model.summary()

model.fit(x_train, yc_train, batch_size=BS, epochs=100)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import seaborn as sn

y_pred = model.predict(x_test)
y_pred = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index = [str(i) for i in range(10)],
                  columns = [str(i) for i in range(10)])

sn.heatmap(df_cm, annot=True, fmt="d")
print(classification_report(y_test, y_pred))

## El problema del XOR:
La función booleana XOR (or exclusivo) no puede ser aprendida por una regresión lógistica.

| $X_0$ | $X_1$ | $Y$ |
| --- | --- | --- |
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Definamos nuestro dataset para probar nuestra RNA.

In [ ]:
x = np.asarray([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.asarray([0, 1, 1, 0])

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x[:, 0], x[:, 1], c=['green' if i==1 else 'red' for i in y])

Generamos un regresor logístico usando una RNA de una neurona, con función de activación sigmoide, una función de pérdida del tipo binary cross-entropy, y en esta oportunidad emplearemos un optimizador adam.

Entrenaremos por 1000 epochs.

In [ ]:
i = Input((2,)) #las entradas son las coordenadas de los puntos
d = Dense(1, activation='sigmoid')(i)

model = Model(i, d)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

h = model.fit(x, y, epochs=1000, verbose=0)

plt.title('Función de pérdida')
plt.xlabel('Epochs')
plt.ylabel('Binary Crossentropy')
plt.plot(h.history['loss'])
plt.show()

plt.title('Metrica accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.plot(h.history['accuracy'])
plt.show()
print(model.predict(x))

Como se puede observar en los gráficos, una sólo  perceptron no puede resolver la tarea.

***
## Perceptrón multicapas

¿Qué pasa si usamos más neuronas?

La red neuronal más sencilla, conocida como perceptrón multicapa (MLP), no es más que capas de perceptrones apilados unas a continuación de otras.

Entonces un MLP tiene la siguiente forma ($l_{i}$ es la capa):

$$l_{1}=f_{1}(\overline{x} \cdot W_{1} + \overline{bias_{1}})$$

$$l_{2}=f_{2}(\overline{l_{1}} \cdot W_{2} + \overline{bias_{2}})$$

$$...$$

$$l_{N}=f_{N}(\overline{l_{N-1}} \cdot W_{N} + \overline{bias_{N}})$$

Es importante destacar que, dada una función de pérdida, calcular el gradiente para cada parámetro de la red ($W_{i}$ o $bias_{i}$), es simplemente aplicar la regla de la cadena en repetidas ocaciones. 

Estos cálculos pueden ser realizados de forma automática por librerias como TensorFlow.

In [ ]:
i = Input((2,))   #entradas
d = Dense(30, activation='tanh')(i)  #capa de 30 neuronas, con función de activación tangente hiperbólica
net = Dense(1, activation='sigmoid')(d) #capa de 1 neurona, con función de activación sigmoide.

model = Model(i, net)
model.compile(loss='binary_crossentropy', optimizer='nadam')

h = model.fit(x, y, epochs=800, verbose=0)

plt.title('Función de pérdida')
plt.xlabel('Epochs')
plt.ylabel('Binary Crossentropy')
plt.plot(h.history['loss'])
plt.show()

for xi, yi in zip(x.tolist(), model.predict(x)[:, 0].tolist()):
    print('{} xor {} -> {}'.format(*xi, yi))

***

### **Ejercicio C-1:**

En el siguiente código aplicaremos una red MLP para realizar la clasificación multiclase del dataset MNIST.

In [ ]:
i = Input((x_train.shape[1],))  #784 entradas
print(x_train.shape)  #60000 muestras x 784
print(y_train.shape)  #60000 muestras 

d = Dense(200)(i)
d = Dense(10, activation='softmax')(d)

model = Model(i, d)

# model.compile(loss='sparse_categorical_crossentropy', optimizer='sgd')  #se usa SCCE cuando mis target son numéricos, no están codificados
model.compile(loss='categorical_crossentropy', optimizer='sgd')  #se usa CCE cuando los target están codificados (one-hot encode) 

model.summary()

# model.fit(x_train, y_train, batch_size=batch_size, epochs=100) #y_train (target numéricos)
model.fit(x_train, yc_train, batch_size=BS, epochs=100)  #yc_train (target codificados)


In [ ]:
y_pred = model.predict(x_test)
y_pred = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index = [str(i) for i in range(10)],
                  columns = [str(i) for i in range(10)])

sn.heatmap(df_cm, annot=True, fmt="d")

print(classification_report(y_test, y_pred))

***
## Optimizadores:

Seleccionar el optimizador y sus hiperparámetros tiene un impacto en CUANTO más rápido aprende nuestro modelo. 

Además del stochastic gradient descent (SGD), exiten otras técnicas de optimización basadas en gradiente. 

[Keras](https://keras.io/api/optimizers/) provee varias, las más comúnes son:

* SGD
* RMSprop
* Adam
* Adadelta
* Adagrad
* Adamax
* Nadam

Desafortunadamente, no exite una técnica que sea superior a todas las otras. No hay consenso en cuál es la más adecuada para cada tipo de problema. 

En general, las técnicas de gradiente adaptativos como Adam, RMSProp y Adagrad funcionan bien en la mayoría de los casos.

**Lecturas recomendadas** 

* Capítulo 8 (sección 8.5.4) de [Deep Learning](https://www.deeplearningbook.org/) de Ian Goodfellow and Yoshua Bengio and Aaron Courville. (2016)
* Schaul, Tom, Ioannis Antonoglou, and David Silver. "Unit tests for stochastic optimization." [arXiv preprint arXiv:1312.6055](https://arxiv.org/abs/1312.6055) (2013).

Utilicemos ahora el optimizador SGD, seteando dos parámetros adicionales:  momentum y Nesterov

In [ ]:
from tensorflow.keras.optimizers import SGD

i = Input((x_train.shape[1],))
d = Dense(200)(i)
d = Dense(10, activation='softmax')(d)

model = Model(i, d)
model.compile(loss='sparse_categorical_crossentropy', optimizer=(SGD(momentum=0.9, nesterov=True)))
model.summary()

model.fit(x_train, y_train, batch_size=BS, epochs=100)

In [ ]:
y_pred = model.predict(x_test)
y_pred = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index = [str(i) for i in range(10)],
                  columns = [str(i) for i in range(10)])

sn.heatmap(df_cm, annot=True, fmt="d")

print(classification_report(y_test, y_pred))

`NOTAS`:

El método de optimización SGD con Nesterov, también conocido como SGD con momento de Nesterov o Nesterov accelerated gradient (NAG), es una variante del algoritmo SGD que introduce un término de momento mejorado para mejorar la convergencia y la velocidad de entrenamiento.

En el SGD tradicional, el momento se aplica en función del gradiente actual para actualizar los parámetros del modelo. Sin embargo, en SGD con Nesterov, el momento se aplica anticipando el próximo paso del gradiente en función de su dirección actual. Esto se logra al calcular el gradiente utilizando una estimación avanzada de los parámetros en lugar de los valores actuales.

El proceso de actualización en SGD con Nesterov se divide en dos etapas:

Primer paso de momento: se calcula una estimación avanzada de los parámetros moviéndose en la dirección del momento. Esto se logra aplicando el momento a los parámetros actuales y calculando el gradiente utilizando esos parámetros anticipados.

Segundo paso de corrección: se realiza una corrección utilizando el gradiente calculado en el paso anterior y los parámetros actuales para actualizar los parámetros finales del modelo.

En resumen, mientras que el método del momentum se basa en acumular el gradiente anterior para guiar el proceso de actualización, el método de Nesterov va un paso más allá y utiliza una estimación anticipada de los parámetros para calcular el gradiente. Esto le permite ajustar de manera más precisa y rápida en regiones difíciles. Ambas técnicas son variantes populares del SGD y se utilizan para mejorar la convergencia y el rendimiento del entrenamiento de modelos de aprendizaje automático.

***
## Framework PyTorch

En el siguiente ejemplo veremos como implementar la misma red utilizando otro framewrok diferente. En lugar de TensorFlow/Keras usaremos [PyTorch](https://pytorch.org/). 

Si bien tiene similitudes, también tiene muchas diferencias.

Primeramente importamos *torch* y preguntamos si tenemos una GPU (Graphics Processing Unit) disponible y el toolkit de CUDA (Compute Unified Device Architecture) instalado, lo cual nos permitirá hacer cómputo paralelo.

In [ ]:
import torch
from torch import nn

#Selecciona el dispositivo
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f'device: {device}')

Como segundo paso definimos la red a utilizar. En este caso será una red con dos capas. A diferencia de la definición anterior, la red no computará la softmax, ya que la misma es computada por la función de perdida.



In [ ]:
class MLP(nn.Module): #definimos una clase perceptron multicapa
    def __init__(self, input_size, classes):
        super(MLP, self).__init__()
        print(f'input_size: {input_size}')
        print(f'num classes: {classes}')
        self.fc1 = nn.Linear(input_size, 200)  #1er capa de 200 neuronas
        self.fc2 = nn.Linear(200, classes)  #2da capa de 10 neuronas (una por cada clase)

    def forward(self, x):
        out1 = self.fc1(x)
        out2 = self.fc2(out1)
        return out2

model = MLP(x_train.shape[1], 10)

### Dataloarders

Definimos los DataLoaders, si bien no son esenciales, ahorran el trabajo de dividir los datos en batches y hacer el shuffle cuando es necesario.

En PyTorch, los DataLoaders son objetos que facilitan la carga y el procesamiento de datos durante el entrenamiento de modelos de aprendizaje automático. Son parte de la biblioteca PyTorch DataLoader y se utilizan para cargar datos de manera eficiente y en paralelo mientras se entrena un modelo.

Un DataLoader combina un conjunto de datos con varias funcionalidades que facilitan su uso durante el entrenamiento. Estas funcionalidades incluyen:

Muestreo de datos: puede realizar el muestreo de datos de manera automática y aleatoria. Esto es útil para garantizar que el modelo vea una variedad de ejemplos durante el entrenamiento y evita el sesgo debido al orden de los datos.

Carga y preprocesamiento eficiente: puede cargar y preprocesar los datos de manera eficiente utilizando técnicas como la carga en paralelo y el almacenamiento en búfer. Esto es especialmente útil cuando se trabaja con conjuntos de datos grandes o cuando es necesario aplicar transformaciones complejas a los datos antes de enviarlos al modelo (data-augmentation).

Batching: puede agrupar los datos en lotes (batches) de tamaño definido. Esto es beneficioso para el entrenamiento en mini lotes (mini-batch training), donde el modelo se actualiza en función del error promedio de un conjunto de ejemplos, en lugar de ejemplos individuales. El batching mejora la eficiencia del entrenamiento y permite aprovechar la aceleración proporcionada por las unidades de procesamiento gráfico (GPU).


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dl_train = DataLoader(TensorDataset(torch.tensor(x_train), torch.tensor(y_train, dtype=torch.long)), batch_size=BS, shuffle=True)
dl_test = DataLoader(TensorDataset(torch.tensor(x_test), torch.tensor(y_test, dtype=torch.long)), batch_size=BS, shuffle=False)

El siguiente código es el equivalente al método fit del modelo de Keras. Si bien este código es más complejo y en general muy repetitivo, en ciertos casos permite tener más flexibilidad. 

Por ejemplo, es más sencillo definir el entrenamiento para Generative Adversarial Networks (GANs). No significa que con Keras no se pueda hacer lo mismo, solo que en esos casos require más codificación.

In [ ]:
#defino el optimizador a usar
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=False)

#defino la función de pérdida
loss = nn.CrossEntropyLoss()
epochs = 100

model.to(device)

for i in range(epochs):
    losses = []
    print(f'Epoch {i+1}', end='->')
    for x, y in dl_train:
        optimizer.zero_grad() #Resetea los gradientes
        
        #Mueve los datos a GPU (o CPU) y calcula la pérdida
        pred = model(x.to(device))
        current_loss = loss(pred, y.to(device))
        
        #Calcula el gradiente
        current_loss.backward()
        optimizer.step()   #actualiza los valores
        losses.append(current_loss.item())
    print('Loss: {}'.format(np.mean(losses)))

Realizar las predicciones también requiere algo más de código, incluso se tiene que aplicar la función softmax de forma manual para calcular las probabilidades. 

**NOTA:**: Para obtener la clase más probable no es necesario calcular la softmax, pero por completitud del ejemplo se encuentra definida.

In [ ]:
from tqdm.notebook import tqdm

y_pred = []
with torch.no_grad(): #No necesitamos los gradientes!
    for x, _ in tqdm(dl_test):
        pred = model(x.to(device))
        pred = nn.functional.softmax(pred, dim=1).cpu().numpy()
        y_pred.extend(np.argmax(pred, axis=-1).tolist())

Para sorpresa de nadie, el resultado de la clasificación es bastante similar.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index = [str(i) for i in range(10)],
                  columns = [str(i) for i in range(10)])

sn.heatmap(df_cm, annot=True, fmt="d")
print(classification_report(y_test, y_pred))

***
NOTA: 

* A medida que el algoritmo de retropropagación avanza hacia atrás, desde la capa de salida hacia la capa de entrada, los gradientes a menudo se vuelven cada vez más pequeños y se acercan a cero, lo que eventualmente deja los pesos de las capas iniciales o inferiores casi sin cambios. Como resultado, el descenso del gradiente nunca converge al óptimo. Esto se conoce como el problema de **vanishing gradient**.

* Por el contrario, en algunos casos, los gradientes siguen haciéndose más y más grandes a medida que avanza el algoritmo de retropropagación. Esto, a su vez, provoca actualizaciones de peso muy grandes y hace que el descenso del gradiente diverja. Esto se conoce como el problema de **exploding gradient**.